# GRAST — WP07.3 pretraining trên Kaggle

1. Import notebook này; bật **Internet** và GPU trong Settings.
2. Upload `grast-wp07-data-block256-17971c0168e4.zip` thành Kaggle Dataset (có thể để private), rồi **Add Input** dataset đó. ZIP giữ nguyên hoặc đã giải nén đều được hỗ trợ; không upload qua cell tương tác.
3. Lần đầu giữ `SMOKE_ONLY=True` để verify 4 updates.
4. Chọn **Save Version → Save & Run All**, xác nhận GPU. Đây là phiên mới chạy từ đầu; không phụ thuộc biến hoặc file chỉ tồn tại trong phiên chỉnh sửa. Khi phiên nền bắt đầu, có thể đóng trình duyệt.
5. Sau smoke thành công, đặt `SMOKE_ONLY=False` để train đủ 2.000 updates. Có thể train từ đầu; nếu muốn tiếp tục smoke, gắn **Output của version smoke** làm Input và điền `RESUME` trỏ đến `latest.pt` dưới `/kaggle/input/...`.

Log luôn hiển thị từng dòng. Checkpoints/curves/report nằm trong `/kaggle/working/grast-wp07`; cần Notebook Output được lưu để giữ chúng sau phiên. Không coi thư mục working là bản backup ngoài runtime khi phiên bị kill hoặc fail.


In [ ]:
import os
import sys
from pathlib import Path

SMOKE_ONLY = True
RESUME = ""  # Optional: /kaggle/input/<previous-notebook-output>/.../checkpoints/latest.pt
INPUT_BASE = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/grast-wp07")
REPO = Path("/tmp/grast-wp07-source")
REVISION = "aa4be36b33ca025701421ef48715a22b43c4e8db"

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("GRAST requires Python 3.11–3.13")
os.environ.update(PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("Mode:", "smoke 4 updates" if SMOKE_ONLY else "pretrain 2000 updates", flush=True)

In [ ]:
import importlib
import subprocess


def run_logged(command, *, cwd=None):
    print("RUN:", " ".join(map(str, command)), flush=True)
    with (WORK_ROOT / "console.log").open("a", encoding="utf-8") as log:
        with subprocess.Popen(
            list(map(str, command)),
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            bufsize=1,
            env=os.environ.copy(),
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                status = process.wait()
            except BaseException:
                import signal

                process.send_signal(signal.SIGINT)
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.terminate()
                raise
            if status:
                raise subprocess.CalledProcessError(status, command)


if not REPO.exists():
    run_logged(["git", "clone", "https://github.com/CuongBien/GRAST.git", REPO])
run_logged(["git", "fetch", "origin"], cwd=REPO)
run_logged(["git", "checkout", "--detach", REVISION], cwd=REPO)
# Keep Kaggle's installed PyTorch/CUDA image; do not replace it automatically.
run_logged([sys.executable, "-u", "-m", "pip", "install", "--no-deps", "-e", "."], cwd=REPO)
run_logged([sys.executable, "-u", "-m", "pip", "install", "matplotlib>=3.9,<4"], cwd=REPO)
# Editable-install .pth files are not reloaded automatically in an existing kernel.
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
run_logged(["git", "rev-parse", "HEAD"], cwd=REPO)

In [ ]:
import json

import torch


def select_precision(capability_major, bf16_supported):
    return "bfloat16" if capability_major >= 8 and bf16_supported else "float16"


if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator before running this notebook")
torch.cuda.set_device(0)
capability = torch.cuda.get_device_capability(0)
PRECISION = select_precision(capability[0], torch.cuda.is_bf16_supported())
environment = {
    "python": sys.version,
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
    "capability": capability,
    "compiled_architectures": torch.cuda.get_arch_list(),
    "precision": PRECISION,
    "device": "cuda:0",
    "revision": REVISION,
}
print(json.dumps(environment, indent=2), flush=True)
# Real forward/backward probe catches CUDA-image/hardware incompatibility early.
probe = torch.ones((32, 32), device="cuda", requires_grad=True)
with torch.autocast("cuda", dtype=getattr(torch, PRECISION)):
    probe_loss = (probe @ probe).float().square().mean()
probe_loss.backward()
torch.cuda.synchronize()
if not torch.isfinite(probe.grad).all():
    raise RuntimeError("GPU precision probe produced non-finite gradients")
del probe, probe_loss
torch.cuda.empty_cache()
(WORK_ROOT / "kaggle_environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)
print("GPU probe passed. This runner uses one GPU (cuda:0), even if two are available.", flush=True)

## Input tự động, có xác minh checksum

Notebook tìm đúng bundle WP07.3 trong các Input đã gắn, chấp nhận ZIP nguyên bản hoặc cây thư mục đã giải nén. Không tải lại corpus và không fit tokenizer. Mọi payload được kiểm tra SHA256, dataset/tokenizer identities trước train. Nếu không tìm thấy, hãy Add Input dataset chứa bundle rồi chạy lại.


In [ ]:
import hashlib
import tempfile

from llm_scratch.data.datasets import file_sha256
from llm_scratch.data.training_bundle import FILES, extract_bundle, verify_inputs

BUNDLE_NAME = "grast-wp07-data-block256-17971c0168e4.zip"
BUNDLE_SHA256 = "9f08cdc7ba7c8a4825ce90470fd19fc3ed6da7d6d3bfd742097a389d1aab949d"
MANIFEST_SHA256 = "45e0a81a7dff706d0f789a369e135d70064d6db14bc9b4c065a2f2e601c509d9"
DATASET_FINGERPRINT = "17971c0168e4367b183ebb0a3abc7c1309ffe5095cdc67899bce13ffc4298faa"


def verify_input_tree(root):
    raw = (root / "bundle_manifest.json").read_bytes()
    if hashlib.sha256(raw).hexdigest() != MANIFEST_SHA256:
        raise ValueError("Expected the frozen WP07.3 block256 bundle manifest")
    manifest = json.loads(raw)
    if set(manifest["files"]) != FILES:
        raise ValueError("Unexpected bundle payload list")
    for name in sorted(FILES):
        path = root / name
        if not path.resolve().is_relative_to(root.resolve()):
            raise ValueError("Bundle payload escapes its input root")
        if file_sha256(path) != manifest["files"][name]:
            raise ValueError("Input checksum mismatch: " + name)
    meta = verify_inputs(root / "pretraining_data", root / "tokenizer")
    if meta["fingerprint"] != DATASET_FINGERPRINT or meta["block_size"] != 256:
        raise ValueError("Wrong dataset/block_size for the pinned recipe")
    return root


def find_inputs(input_base):
    for path in sorted(input_base.rglob("bundle_manifest.json")):
        if hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST_SHA256:
            return verify_input_tree(path.parent)
    archives = sorted(input_base.rglob(BUNDLE_NAME))
    if archives:
        root = Path(tempfile.mkdtemp(prefix="grast-kaggle-inputs-")) / "verified"
        extract_bundle(archives[0], root, expected_sha256=BUNDLE_SHA256)
        # Extractor validates payloads; record the same known manifest identity for both layouts.
        # extract_bundle omits the transport manifest from its output.
        return root
    raise FileNotFoundError(
        "Add Input containing " + BUNDLE_NAME + " or its extracted bundle_manifest.json"
    )


INPUTS = find_inputs(INPUT_BASE)
print("Verified inputs:", INPUTS, flush=True)

## Train và checkpoint

Recipe giữ nguyên model46,15M, block256/B2/G8, budget2.000 updates. T4/P100 chọn FP16; GPU có BF16 native chọn BF16. Hardware phải tương thích với bản PyTorch của runtime; lỗi kernel được báo ở probe, không fallback CPU. Không có DDP/multi-GPU ở WP07.3.

Nếu `RESUME` rỗng, runner khởi tạo từ đầu với seed42. Muốn tiếp tục output phiên trước, gắn output đó làm Input và dùng đường dẫn checkpoint dưới `/kaggle/input/...`; Save & Run All không lấy các file nháp của phiên editor. Resume giữ strict recipe/precision/GPU/Torch/runtime contract; không ép đổi BF16↔FP16 hoặc Colab↔Kaggle.


In [ ]:
def training_command(repo, inputs, output_root, precision, smoke_only, resume):
    command = [
        sys.executable,
        "-u",
        "-m",
        "llm_scratch.training.pretrain",
        "--config",
        repo / "configs/training/wp07_pretrain.toml",
        "--prepared-root",
        inputs / "pretraining_data",
        "--tokenizer",
        inputs / "tokenizer",
        "--device",
        "cuda",
        "--precision",
        precision,
        "--output-root",
        output_root,
    ]
    if smoke_only:
        command += ["--stop-after", "4"]
    if resume:
        path = Path(resume)
        if not path.is_file():
            raise FileNotFoundError("Attach the previous notebook Output as Input: " + resume)
        command += ["--resume", path]
    return command


OUTPUT_ROOT = WORK_ROOT / "runs"
run_logged(training_command(REPO, INPUTS, OUTPUT_ROOT, PRECISION, SMOKE_ONLY, RESUME), cwd=REPO)

In [ ]:
import zipfile

from IPython.display import FileLink, Image, display

RUN = sorted(p for p in OUTPUT_ROOT.iterdir() if p.is_dir())[-1]
print("Run:", RUN, flush=True)
print("Checkpoint:", RUN / "checkpoints/latest.pt", flush=True)
report = json.loads((RUN / "artifacts/pretraining_report.json").read_text(encoding="utf-8"))
print(
    json.dumps(
        {
            k: report[k]
            for k in ("status", "step", "planned_steps", "tokens_optimized", "dataset_fingerprint")
        },
        indent=2,
    ),
    flush=True,
)
display(Image(filename=str(RUN / "artifacts/loss_curves.png")))
SUMMARY = WORK_ROOT / (RUN.name + "-summary.zip")
with zipfile.ZipFile(SUMMARY, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and path.suffix not in {".pt", ".u32"}:
            archive.write(path, path.relative_to(RUN))
display(FileLink(str(SUMMARY)))
print("Keep raw checkpoints in Notebook Output; the summary ZIP excludes weights.", flush=True)

## Sau khi phiên nền hoàn tất

Xem logs/output của version đã chạy; tải summary ZIP và giữ `runs/.../checkpoints/` trong Notebook Output cho WP07.4/5. Mỗi full checkpoint khoảng554MB; latest + các mốc + final khoảng2,8GB cho một full run. Mặc định smoke4 chưa phải baseline hoàn chỉnh.

Code notebook và input paths được verify local; chưa có bằng chứng thực thi trên Kaggle/T4/P100. Không áp ước tính thời gian A100 trực tiếp cho Kaggle. Sau100 updates đầu, dùng elapsed logs để dự đoán thời gian còn lại.
